<a href="https://colab.research.google.com/github/adalbertii/LLMs/blob/main/gpt_model_creation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 1: Praca z tekstem

In [ ]:
!pip install tiktoken

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 14.3 MB/s eta 0:00:00


In [ ]:
from importlib.metadata import version
import tiktoken
import torch
from torch.utils.data import Dataset, DataLoader


In [ ]:
pkgs = ["matplotlib",
        "numpy",
        "tiktoken",
        "torch",
        "tensorflow" # For OpenAI's pretrained weights
       ]
for p in pkgs:
    print(f"{p} version: {version(p)}")

matplotlib version: 3.8.0
numpy version: 1.26.4
tiktoken version: 0.8.0
torch version: 2.5.1+cu121
tensorflow version: 2.17.1


## Tokenizacja tekstu

- załadowanie pliku do środowiska CoLab
- "The Verdict by Edith Wharton" (https://en.wikisource.org/wiki/The_Verdict) jest publicznie dostępnym dataset-em

In [ ]:
with open("the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

print("Całkowita liczba znaków w dokumencie:", len(raw_text))
print(raw_text[:99])

Całkowita liczba znaków w dokumencie: 20479
I HAD always thought Jack Gisburn rather a cheap genius--though a good fellow enough--so it was no 


- Celem jest tokenizacja oraz embedding tego tekstu aby móc zasilić budowany model LLM
- Najpierw przeprowadzę próby budowy prostego tokenizatora opartego na przykładowym tekście


In [ ]:
import re

text = "Hello, world. This, is a test."
# podział na tokenty z wykorzystaniem wyrazen regularnych (odstęp jako znacznik podziału)
result = re.split(r'(\s)', text)

print(result)

['Hello,', ' ', 'world.', ' ', 'This,', ' ', 'is', ' ', 'a', ' ', 'test.']


- We don't only want to split on whitespaces but also commas and periods, so let's modify the regular expression to do that as well

In [ ]:
# jesli uwzglednimy w wyrazeniu regularnym równiez kropki i przecinki:
result = re.split(r'([,.]|\s)', text)

print(result)

['Hello', ',', '', ' ', 'world', '.', '', ' ', 'This', ',', '', ' ', 'is', ' ', 'a', ' ', 'test', '.', '']


In [ ]:
# Usuniecie białe znaki z każdego elementu, a następnie odfiltrowanie  puste ciągi
result = [item for item in result if item.strip()]
print(result)

['Hello', ',', 'world', '.', 'This', ',', 'is', 'a', 'test', '.']


- jeśli rozszerzymy wyrażenia regularne na jeszcze inne znaki interpunkcyjne

In [ ]:
text = "Hello, world. Is this-- a test?"

result = re.split(r'([,.:;?_!"()\']|--|\s)', text)
result = [item.strip() for item in result if item.strip()]
print(result)

['Hello', ',', 'world', '.', 'Is', 'this', '--', 'a', 'test', '?']


- zastosujemy ten typ tokenizacji do surowego tekstu odczytanego z pliku "the-verdict.txt"

In [ ]:
preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', raw_text)
preprocessed = [item.strip() for item in preprocessed if item.strip()]
print(preprocessed[:30])

['I', 'HAD', 'always', 'thought', 'Jack', 'Gisburn', 'rather', 'a', 'cheap', 'genius', '--', 'though', 'a', 'good', 'fellow', 'enough', '--', 'so', 'it', 'was', 'no', 'great', 'surprise', 'to', 'me', 'to', 'hear', 'that', ',', 'in']


- Let's calculate the total number of tokens

In [ ]:
# całkowita liczba tokenów w dokumencie wynosi:
print(len(preprocessed))

4690


## Konwersja otrzymanych tokenów do tzw. IDs ów

- Z tokenów tworzymy tzw. korpus składający się z unkalnych tokenów

In [ ]:
all_words = sorted(set(preprocessed))
vocab_size = len(all_words)

print(vocab_size)

1130


In [ ]:
# tworzywmy słownik mapując każdy unikalny token do liczby (integer)
vocab = {token:integer for integer,token in enumerate(all_words)}

In [ ]:
# pierwsze 20 pozycji w słowniku
for i, item in enumerate(vocab.items()):
    print(item)
    if i >= 20:
        break

('!', 0)
('"', 1)
("'", 2)
('(', 3)
(')', 4)
(',', 5)
('--', 6)
('.', 7)
(':', 8)
(';', 9)
('?', 10)
('A', 11)
('Ah', 12)
('Among', 13)
('And', 14)
('Are', 15)
('Arrt', 16)
('As', 17)
('At', 18)
('Be', 19)
('Begin', 20)


- Ilustracja tokenizacji przykładowego tekstu

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch02_compressed/07.webp?123" width="500px">

- Zbierając powyższe przekształcenia w calość implementuję klasę SimpleTokenizerV1

In [ ]:
class SimpleTokenizerV1:
    def __init__(self, vocab):
        # vocab - słownik
        self.str_to_int = vocab
        self.int_to_str = {i:s for s,i in vocab.items()}

    # dzieli tekst na tokeny i tworzy ich indeksy (IDs -y )
    def encode(self, text):
        preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', text)

        preprocessed = [
            item.strip() for item in preprocessed if item.strip()
        ]
        ids = [self.str_to_int[s] for s in preprocessed]
        return ids

    # zamienia IDs tokenów w teskt
    def decode(self, ids):
        text = " ".join([self.int_to_str[i] for i in ids])
        # Replace spaces before the specified punctuations
        text = re.sub(r'\s+([,.?!"()\'])', r'\1', text)
        return text

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch02_compressed/08.webp?123" width="500px">

- Możemy użyć tokenizera do kodowania (czyli tokenizowania) tekstów na liczby całkowite IDs
- Tel liczby (IDs-y) mogą zostać poddane procesowi embedding-u i przekazane jako dane wejściowe do modelu LLM

In [ ]:
tokenizer = SimpleTokenizerV1(vocab)

text = """"It's the last he painted, you know,"
           Mrs. Gisburn said with pardonable pride."""
ids = tokenizer.encode(text)
print(ids)

[1, 56, 2, 850, 988, 602, 533, 746, 5, 1126, 596, 5, 1, 67, 7, 38, 851, 1108, 754, 793, 7]


- Możemy zdekodować te IDs- y od tekstu

In [ ]:
tokenizer.decode(ids)

'" It\' s the last he painted, you know," Mrs. Gisburn said with pardonable pride.'

In [ ]:
tokenizer.decode(tokenizer.encode(text))

'" It\' s the last he painted, you know," Mrs. Gisburn said with pardonable pride.'

## Dodanie specjalych tokenów kontekstowych

- dodajemy specalne tokeny dla nieznanych wyrazów oraz tokenów odpowiadajacych końcowi przetwarzanego tekstu

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch02_compressed/09.webp?123" width="500px">

- Niektóre ze specjalnych tokentów kontekstowych:
  - `[BOS]` (początek sekwencji) oznacza początek tekstu
  - `[EOS]` (koniec sekwencji) oznacza miejsce, w którym kończy się tekst (jest to zwykle używane do łączenia wielu niepowiązanych ze sobą tekstów, np. dwóch różnych artykułów w Wikipedii lub dwóch różnych książek itd.)
  - `[PAD]` (dopełnienie), jeśli szkolimy LLM o rozmiarze partii większym niż 1 (możemy dołączyć wiele tekstów o różnej długości; za pomocą tokena dopełnienia dopełniamy krótsze teksty do najdłuższej długości, tak aby wszystkie teksty miały tę samą długość)
- `[UNK]` do reprezentowania słów, które nie są zawarte w słowniku



- GPT-2 nie potrzebuje żadnego z tych tokenów wymienionych powyżej, a jedynie używa `<|endoftext|>`
- `<|endoftext|>` jest analogiczny do `[EOS]`
- GPT używa również `<|endoftext|>` do dopełnienia (ponieważ zazwyczaj używamy maski podczas uczenia na danych wejściowych wsadowych, i tak nie będziemy brać pod uwagę dopełnionych tokenów, więc nie ma znaczenia, jakie to są te tokeny)
- GPT-2 nie używa tokenu `<UNK>` dla nieznanych wyrazów; zamiast tego GPT-2 wykorzystuje tokenizator (BPE), który dzieli słowa na jednostki podsłów



- Używamy tokenów <|endoftext|> pomiędzy dwoma niezależnymi źródłami tekstu:

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch02_compressed/10.webp" width="500px">

- rozszerzmy nasz korpus o dwa specjalne tokeny <|endoftext|>" i "<|unk|>":

In [ ]:
all_tokens = sorted(list(set(preprocessed)))
all_tokens.extend(["<|endoftext|>", "<|unk|>"])

vocab = {token:integer for integer,token in enumerate(all_tokens)}

In [ ]:
len(vocab.items())

1132

- poprzedni rozmiar korpusu wynosił: 1130 tokenów

In [ ]:
for i, item in enumerate(list(vocab.items())[-5:]):
    print(item)

('younger', 1127)
('your', 1128)
('yourself', 1129)
('<|endoftext|>', 1130)
('<|unk|>', 1131)


- dostosowuje klasętokenizatora do obsługi nieznanych tokenów `<unk>` (nie istniejąych w korpusie)

In [ ]:
class SimpleTokenizerV2:
    def __init__(self, vocab):
        self.str_to_int = vocab
        self.int_to_str = { i:s for s,i in vocab.items()}

    def encode(self, text):
        preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', text)
        preprocessed = [item.strip() for item in preprocessed if item.strip()]
        preprocessed = [
            item if item in self.str_to_int
            else "<|unk|>" for item in preprocessed
        ]

        ids = [self.str_to_int[s] for s in preprocessed]
        return ids

    def decode(self, ids):
        text = " ".join([self.int_to_str[i] for i in ids])

        text = re.sub(r'\s+([,.:;?!"()\'])', r'\1', text)
        return text

Test nowej wersji tokenizatora :

In [ ]:
tokenizer = SimpleTokenizerV2(vocab)

text1 = "Hello, do you like tea?"
text2 = "In the sunlit terraces of the palace."

text = " <|endoftext|> ".join((text1, text2))

print(text)

Hello, do you like tea? <|endoftext|> In the sunlit terraces of the palace.


In [ ]:
tokenizer.encode(text)

[1131, 5, 355, 1126, 628, 975, 10, 1130, 55, 988, 956, 984, 722, 988, 1131, 7]

In [ ]:
# dekodowanie IDs ów
tokenizer.decode(tokenizer.encode(text))

'<|unk|>, do you like tea? <|endoftext|> In the sunlit terraces of the <|unk|>.'

## Kodowanie typu BytePair encoding [BPE]

- GPT-2 wykorzystuje kodowanie BytePair (BPE) jako swój tokenizator
- pozwala modelowi podzielić słowa, które nie znajdują się w jego predefiniowanym słownictwie, na mniejsze jednostki podsłów lub nawet pojedyncze znaki, umożliwiając obsługę słów spoza słownika
- tutaj użyję tokenizera BPE z otwartej biblioteki tiktoken OpenAI, która implementuje swoje podstawowe algorytmy w Rust w celu poprawy wydajności obliczeniowej
   

In [ ]:
# pip install tiktoken

In [ ]:
import importlib
import tiktoken

print("tiktoken version:", importlib.metadata.version("tiktoken"))

tiktoken version: 0.8.0


In [ ]:
tokenizer = tiktoken.get_encoding("gpt2")

In [ ]:
text = (
    "Hello, do you like tea? <|endoftext|> In the sunlit terraces"
     "of someunknownPlace."
)

integers = tokenizer.encode(text, allowed_special={"<|endoftext|>"})

print(integers)

[15496, 11, 466, 345, 588, 8887, 30, 220, 50256, 554, 262, 4252, 18250, 8812, 2114, 1659, 617, 34680, 27271, 13]


In [ ]:
strings = tokenizer.decode(integers)

print(strings)

Hello, do you like tea? <|endoftext|> In the sunlit terracesof someunknownPlace.


- Tokenizatory BPE dzielą nieznane słowa na podsłowa i pojedyncze znaki:

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch02_compressed/11.webp" width="300px">

## Próbkowanie danych z przesuwanym oknem [  sampling with a sliding window]

- Szkolimy LLM, aby generowały jedno słowo na raz, dlatego chcemy odpowiednio przygotować dane szkoleniowe, gdzie następne słowo w sekwencji reprezentuje przewidywany cel (label dla procesu predykcji):

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch02_compressed/12.webp" width="400px">

In [ ]:
with open("the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

enc_text = tokenizer.encode(raw_text)
print(len(enc_text))

5145


- Dla każdego fragmentu tekstu potrzebujemy danych wejściowych i celów (label)
- Ponieważ chcemy, aby model przewidywał następne słowo, celem są dane wejściowe przesunięte o jedną pozycję w prawo

In [ ]:
enc_sample = enc_text[50:]

In [ ]:
print(len(enc_sample))

5095


In [ ]:
context_size = 4

x = enc_sample[:context_size]
y = enc_sample[1:context_size+1]

print(f"x: {x}")
print(f"y:      {y}")

x: [290, 4920, 2241, 287]
y:      [4920, 2241, 287, 257]


- oczekiwane wyniki predykcji (label) jeden po drugim będą wyglądać następująco:

In [ ]:
for i in range(1, context_size+1):
    context = enc_sample[:i]
    desired = enc_sample[i]

    print(context, "---->", desired)

[290] ----> 4920
[290, 4920] ----> 2241
[290, 4920, 2241] ----> 287
[290, 4920, 2241, 287] ----> 257


In [ ]:
for i in range(1, context_size+1):
    context = enc_sample[:i]
    desired = enc_sample[i]

    print(tokenizer.decode(context), "---->", tokenizer.decode([desired]))

 and ---->  established
 and established ---->  himself
 and established himself ---->  in
 and established himself in ---->  a


- Implementujemy prosty moduł ładujący dane, który iteruje po wejściowym zbiorze danych i zwraca dane wejściowe i cele predykcji [label-e] przesunięte o jeden

In [ ]:
import torch
print("PyTorch version:", torch.__version__)

PyTorch version: 2.5.1+cu121


- Stosujemy metodę sliding window , zmieniając pozycję o +1 :

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch02_compressed/13.webp?123" width="500px">

- Implementuje klasę opisującą dataset oraz moduł ładujący dane, które wyodrębnią fragmenty ze zbioru danych tekstu wejściowego



In [ ]:
from torch.utils.data import Dataset, DataLoader


class GPTDatasetV1(Dataset):
    def __init__(self, txt, tokenizer, max_length, stride):
        self.input_ids = []
        self.target_ids = []

        # tokenizacja całego tekstu (txt)
        token_ids = tokenizer.encode(txt, allowed_special={"<|endoftext|>"})

        # wykorzystanie sliding window do dzielenia ktekstu na nakładające się sekwencje o maksymalnej długości  max_length
        # stride=1 - przesunięcie o jeden token
        for i in range(0, len(token_ids) - max_length, stride):
            input_chunk = token_ids[i:i + max_length]
            target_chunk = token_ids[i + 1: i + max_length + 1]
            self.input_ids.append(torch.tensor(input_chunk))
            self.target_ids.append(torch.tensor(target_chunk))

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        return self.input_ids[idx], self.target_ids[idx]

In [ ]:
# definicja modułu łądującego dane
def create_dataloader_v1(txt, batch_size=4, max_length=256,
                         stride=128, shuffle=True, drop_last=True,
                         num_workers=0):

    # inicjalizacja tokenizatora
    tokenizer = tiktoken.get_encoding("gpt2")

    # utworzenie dataset-u
    dataset = GPTDatasetV1(txt, tokenizer, max_length, stride)

    # stworzenie isntancji modułu ładującego
    dataloader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        drop_last=drop_last,
        num_workers=num_workers
    )

    return dataloader

- Test moduł ładujący dane z rozmiarem batch = 1 oraz kotekstem o długości 4 tokenów :

In [ ]:
with open("the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

In [ ]:
dataloader = create_dataloader_v1(
    raw_text, batch_size=1, max_length=4, stride=1, shuffle=False
)

# zainicjowanie iteratora
data_iter = iter(dataloader)

# pierwszy batch
first_batch = next(data_iter)
print(first_batch)

[tensor([[  40,  367, 2885, 1464]]), tensor([[ 367, 2885, 1464, 1807]])]


- pierwszy tensor jest X-em, drugi Y-kiem (labelem) w procesie póżniejszego uczenia modelu

In [ ]:
second_batch = next(data_iter)
print(second_batch)

[tensor([[ 367, 2885, 1464, 1807]]), tensor([[2885, 1464, 1807, 3619]])]


- przykład wykorzystujący stride = długosci konstekstu (tu 4) :

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch02_compressed/14.webp" width="500px">

- Możemy również tworzyć batch-e o rozmiarze 8
- zwiększam tutaj krok pzresunięcia (stride) , aby uniknąć nakładania się partii, ponieważ większe nakładanie się może prowadzić do zwiększonego nadmiernego dopasowania (overfitting)



In [ ]:
dataloader = create_dataloader_v1(raw_text, batch_size=8, max_length=4, stride=4, shuffle=False)

data_iter = iter(dataloader)
inputs, targets = next(data_iter)
print("Wejscie:\n", inputs)
print("\nOczekiwane wyjście:\n", targets)

Wejscie:
 tensor([[   40,   367,  2885,  1464],
        [ 1807,  3619,   402,   271],
        [10899,  2138,   257,  7026],
        [15632,   438,  2016,   257],
        [  922,  5891,  1576,   438],
        [  568,   340,   373,   645],
        [ 1049,  5975,   284,   502],
        [  284,  3285,   326,    11]])

Oczekiwane wyjście:
 tensor([[  367,  2885,  1464,  1807],
        [ 3619,   402,   271, 10899],
        [ 2138,   257,  7026, 15632],
        [  438,  2016,   257,   922],
        [ 5891,  1576,   438,   568],
        [  340,   373,   645,  1049],
        [ 5975,   284,   502,   284],
        [ 3285,   326,    11,   287]])


## Proces embeddingu

- embedding czyli osadzenie tokenów w ciągłej reprezentacji wektorowej za pomocą warstwy osadzającej
- zwykle te warstwy osadzające są częścią samego LLM i są aktualizowane (uczone) podczas uczenia modelu



<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch02_compressed/15.webp" width="400px">

- załóżmy, że mamy tensor wejściowy o długości 4 z następujacymy IDs-ami (uzyskanymi w procesie tokenizacji) :

In [ ]:
input_ids = torch.tensor([2, 3, 5, 1])

- załózmy również , że mamy bardzo mały korpus (słownik) składajacy się tylko z 6 wyrazów i chcemy poddać go procesowi embedding-owemu (przyjmując, że długość wektora embeddingowego będzie wynosiła 3):

In [ ]:
vocab_size = 6
output_dim = 3

torch.manual_seed(123)
embedding_layer = torch.nn.Embedding(vocab_size, output_dim)

- w rezultacie otrzymujemy macierz 6x3 wag:

In [ ]:
print(embedding_layer.weight)

Parameter containing:
tensor([[ 0.3374, -0.1778, -0.1690],
        [ 0.9178,  1.5810,  1.3010],
        [ 1.2753, -0.2010, -0.1606],
        [-0.4015,  0.9666, -1.1481],
        [-1.1589,  0.3255, -0.6315],
        [-2.8400, -0.7849, -1.4096]], requires_grad=True)


embedding_layer -można  postrzegać jako warstwę sieci neuronowej, którą można zoptymalizować poprzez propagację wsteczną



- jeśli poddamy procesowi embeddingu token o np. IDs = 3 (z przykładu input_ids), ustawijąc długość wynikoweg wektora embeddingu na 3 to otrzymamy :

In [ ]:
print(embedding_layer(torch.tensor([3])))

tensor([[-0.4015,  0.9666, -1.1481]], grad_fn=<EmbeddingBackward0>)


- w wyniku otrzymujemy 4 wiersz z macierzy embeddingu (zmienna embedding_layer.weight)

In [ ]:
# poddając embeddingowi cały wektor input_ids  otrzymujemy:
print(embedding_layer(input_ids))

tensor([[ 1.2753, -0.2010, -0.1606],
        [-0.4015,  0.9666, -1.1481],
        [-2.8400, -0.7849, -1.4096],
        [ 0.9178,  1.5810,  1.3010]], grad_fn=<EmbeddingBackward0>)


- Warstwa osadzająca embedding_layer jest zasadniczo operacją przeglądania matrycy embeddingowej :

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch02_compressed/16.webp?123" width="500px">

## Kodowanie pozycji wyrazu w tekście

- Embedding layer konwertuje identyfikatory IDs na identyczne reprezentacje wektorowe, niezależnie od tego, gdzie się znajdują w sekwencji wejściowej:

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch02_compressed/17.webp" width="400px">

- pozycyjne embeddings są łączone z wektorem osadzania tokenu, tworząc osadzania wejściowe dla dużego modelu językowego LLM:

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch02_compressed/18.webp" width="500px">

- koder  BytePair , w przypadku gpt-2 ma rozmiar korpusu:  50 257
- załózmy, że chcemy zakodować tokeny wejściwe w  wektory embedding-owe o rozmiarze 256

In [ ]:
vocab_size = 50257
output_dim = 256

token_embedding_layer = torch.nn.Embedding(vocab_size, output_dim)

- jeśli przyjmiemy rozmiar  batch size na 8 , a każy wektor IDs-owy ma długość , np. 4 , w rezultacie otrzymamy tensor wo wymiarach 8 x 4 x 256 :

In [ ]:
max_length = 4
dataloader = create_dataloader_v1(
    raw_text, batch_size=8, max_length=max_length,
    stride=max_length, shuffle=False
)
data_iter = iter(dataloader)
inputs, targets = next(data_iter)

In [ ]:
print("Token IDs:\n", inputs)
print("\nInputs shape:\n", inputs.shape)

Token IDs:
 tensor([[   40,   367,  2885,  1464],
        [ 1807,  3619,   402,   271],
        [10899,  2138,   257,  7026],
        [15632,   438,  2016,   257],
        [  922,  5891,  1576,   438],
        [  568,   340,   373,   645],
        [ 1049,  5975,   284,   502],
        [  284,  3285,   326,    11]])

Inputs shape:
 torch.Size([8, 4])


In [ ]:
token_embeddings = token_embedding_layer(inputs)
print(token_embeddings.shape)

torch.Size([8, 4, 256])


- GPT-2 wykorzystuje osadzanie w pozycji bezwzględnej, więc po prostu tworzymy kolejną warstwę osadzania:

In [ ]:
context_length = max_length
pos_embedding_layer = torch.nn.Embedding(context_length, output_dim)

In [ ]:

pos_embeddings = pos_embedding_layer(torch.arange(max_length))
print(pos_embeddings.shape)

torch.Size([4, 256])


- Aby utworzyć osadzenie wejściowe używane w LLM, po prostu dodajemy osadzenia tokenów osadzeni ich pozycji



In [ ]:
input_embeddings = token_embeddings + pos_embeddings
print(input_embeddings.shape)

torch.Size([8, 4, 256])


<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch02_compressed/19.webp" width="400px">